# 🏋️ Challenge 08 — NumPy & Pandas
*Covers Module 18.* 🟢 easy · 🟡 medium · 🔴 hard. Needs `numpy` and `pandas` (`pip install numpy pandas`).

In [1]:
import numpy as np
import pandas as pd

def check(name, test):
    try:
        ok = bool(test())
    except Exception as e:
        ok = False
        name += f"   (raised {type(e).__name__}: {e})"
    print("✅ PASS" if ok else "❌ FAIL", "-", name)

### 🟢 Problem 1 — Array basics
`make_grid()` returns a `3x4` NumPy array of the integers `1..12` (row-major). `border_sum(a)` returns the sum of the **outer border** elements of a 2-D array (no double counting).

In [2]:
def make_grid():
    return np.arange(1, 13).reshape(3, 4)

def border_sum(a):
    return int(a.sum() - a[1:-1, 1:-1].sum())

In [3]:
check("grid shape", lambda: make_grid().shape == (3, 4))
check("grid values", lambda: make_grid()[2, 3] == 12 and make_grid()[0, 0] == 1)
check("border sum of grid", lambda: border_sum(make_grid()) == 78 - 6 - 7)
check("border sum 2x2 is total", lambda: border_sum(np.array([[1, 2], [3, 4]])) == 10)

✅ PASS - grid shape
✅ PASS - grid values
✅ PASS - border sum of grid
✅ PASS - border sum 2x2 is total


### 🟢 Problem 2 — Boolean masks
`clip_outliers(a, low, high)` returns a copy where values `< low` become `low` and values `> high` become `high` (do it with masks, not `np.clip`). `count_between(a, low, high)` counts values with `low <= x <= high`.

In [4]:
def clip_outliers(a, low, high):
    b = a.copy()
    b[b < low] = low
    b[b > high] = high
    return b

def count_between(a, low, high):
    return int(((a >= low) & (a <= high)).sum())

In [5]:
arr = np.array([-5, 3, 12, 7, 99, 0])
check("clip", lambda: clip_outliers(arr, 0, 10).tolist() == [0, 3, 10, 7, 10, 0])
check("original untouched", lambda: arr.tolist() == [-5, 3, 12, 7, 99, 0])
check("count_between", lambda: count_between(arr, 0, 10) == 3)

✅ PASS - clip
✅ PASS - original untouched
✅ PASS - count_between


### 🟢 Problem 3 — Vectorised statistics
`normalize(a)` → z-scores `(a - mean) / std` (population std, NumPy default). `column_means(m)` → mean of each column. `argmax_per_row(m)` → index of the max in each row.

In [6]:
def normalize(a):
    return (a - a.mean()) / a.std()

def column_means(m):
    return m.mean(axis=0)

def argmax_per_row(m):
    return m.argmax(axis=1)

In [7]:
z = normalize(np.array([1.0, 2.0, 3.0, 4.0, 5.0]))
check("normalize mean 0, std 1", lambda: abs(z.mean()) < 1e-12 and abs(z.std() - 1) < 1e-12)
M = np.array([[1, 9, 3], [7, 2, 8]])
check("column_means", lambda: column_means(M).tolist() == [4.0, 5.5, 5.5])
check("argmax_per_row", lambda: argmax_per_row(M).tolist() == [1, 2])

✅ PASS - normalize mean 0, std 1
✅ PASS - column_means
✅ PASS - argmax_per_row


### 🟡 Problem 4 — Broadcasting
`multiplication_table(n)` → an `n x n` array where cell `(i, j)` is `(i+1)*(j+1)`, built with **broadcasting** (no loops). `center_columns(m)` → subtract each column's mean from that column.

<details><summary>💡 Hint</summary>`np.arange(1, n+1)[:, None] * np.arange(1, n+1)` — a column vector times a row vector.</details>

In [8]:
def multiplication_table(n):
    r = np.arange(1, n + 1)
    return r[:, None] * r

def center_columns(m):
    return m - m.mean(axis=0)

In [9]:
check("table 3", lambda: multiplication_table(3).tolist() == [[1, 2, 3], [2, 4, 6], [3, 6, 9]])
check("table shape", lambda: multiplication_table(9).shape == (9, 9))
c = center_columns(np.array([[1.0, 10.0], [3.0, 20.0]]))
check("center_columns", lambda: c.tolist() == [[-1.0, -5.0], [1.0, 5.0]])
check("column means are zero", lambda: np.allclose(c.mean(axis=0), 0))

✅ PASS - table 3
✅ PASS - table shape
✅ PASS - center_columns
✅ PASS - column means are zero


### 🟡 Problem 5 — Linear algebra
Solve the system `2x + y = 5`, `x + 3y = 10` with `solve_system()` (return a NumPy array `[x, y]`). Implement `is_symmetric(m)` and `matrix_power(m, k)` (k ≥ 0, using repeated `@`, not `np.linalg.matrix_power`).

In [10]:
def solve_system():
    A = np.array([[2, 1], [1, 3]])
    b = np.array([5, 10])
    return np.linalg.solve(A, b)

def is_symmetric(m):
    return bool(np.array_equal(m, m.T))

def matrix_power(m, k):
    result = np.eye(len(m), dtype=m.dtype)
    for _ in range(k):
        result = result @ m
    return result

In [11]:
check("solve_system", lambda: np.allclose(solve_system(), [1, 3]))
check("is_symmetric", lambda: is_symmetric(np.array([[1, 2], [2, 1]])) and not is_symmetric(np.array([[1, 2], [3, 1]])))
F = np.array([[1, 1], [1, 0]])
check("matrix_power gives Fibonacci", lambda: matrix_power(F, 10)[0, 1] == 55)
check("power 0 is identity", lambda: matrix_power(F, 0).tolist() == [[1, 0], [0, 1]])

✅ PASS - solve_system
✅ PASS - is_symmetric
✅ PASS - matrix_power gives Fibonacci
✅ PASS - power 0 is identity


### 🟡 Problem 6 — Pandas basics
Build the DataFrame with `make_df()` from this data, then:
- `adults(df)` → rows with `age >= 18`
- `avg_by_city(df)` → Series of mean `score` by `city`, sorted descending
- `add_grade(df)` → new column `grade`: `"pass"` if `score >= 50` else `"fail"` (return a **new** DataFrame; don't mutate the input)

| name | age | city | score |
|---|---|---|---|
| Asha | 21 | Pune | 88 |
| Ravi | 17 | Delhi | 45 |
| Meena | 25 | Pune | 62 |
| Tom | 30 | Delhi | 49 |

In [12]:
def make_df():
    return pd.DataFrame({
        "name": ["Asha", "Ravi", "Meena", "Tom"],
        "age": [21, 17, 25, 30],
        "city": ["Pune", "Delhi", "Pune", "Delhi"],
        "score": [88, 45, 62, 49],
    })

def adults(df):
    return df[df["age"] >= 18]

def avg_by_city(df):
    return df.groupby("city")["score"].mean().sort_values(ascending=False)

def add_grade(df):
    return df.assign(grade=np.where(df["score"] >= 50, "pass", "fail"))

In [13]:
df = make_df()
check("shape", lambda: df.shape == (4, 4))
check("adults", lambda: adults(df)["name"].tolist() == ["Asha", "Meena", "Tom"])
check("avg_by_city", lambda: avg_by_city(df).to_dict() == {"Pune": 75.0, "Delhi": 47.0} and list(avg_by_city(df).index) == ["Pune", "Delhi"])
check("add_grade", lambda: add_grade(df)["grade"].tolist() == ["pass", "fail", "pass", "fail"])
check("input not mutated", lambda: "grade" not in df.columns)

✅ PASS - shape
✅ PASS - adults
✅ PASS - avg_by_city
✅ PASS - add_grade
✅ PASS - input not mutated


### 🟡 Problem 7 — Missing data
`clean(df)` for a DataFrame with columns `price` (numeric with NaNs) and `category` (text with NaNs): fill missing `price` with the **median** price of its `category` (fall back to the overall median if the category itself is missing/all NaN), and fill missing `category` with `"unknown"`. Return a new DataFrame.

<details><summary>💡 Hint</summary>`df.groupby("category")["price"].transform("median")` gives per-row category medians.</details>

In [14]:
def clean(df):
    out = df.copy()
    cat_median = out.groupby("category")["price"].transform("median")
    out["price"] = out["price"].fillna(cat_median).fillna(out["price"].median())
    out["category"] = out["category"].fillna("unknown")
    return out

In [15]:
raw = pd.DataFrame({"price": [10.0, np.nan, 30.0, 20.0, np.nan, 100.0],
                    "category": ["a", "a", "a", "b", None, "b"]})
res = clean(raw)
check("no NaNs left", lambda: not res.isna().any().any())
check("category median used", lambda: res.loc[1, "price"] == 20.0)
check("missing category filled", lambda: res.loc[4, "category"] == "unknown")
check("overall median fallback", lambda: res.loc[4, "price"] == 25.0)
check("original untouched", lambda: raw["price"].isna().sum() == 2)

✅ PASS - no NaNs left
✅ PASS - category median used
✅ PASS - missing category filled
✅ PASS - overall median fallback
✅ PASS - original untouched


### 🟡 Problem 8 — GroupBy & aggregation
Given sales data, `summary(df)` returns a DataFrame indexed by `region` with columns `total` (sum of `amount`), `orders` (row count) and `avg` (mean amount rounded to 2), sorted by `total` descending.

In [16]:
def summary(df):
    out = df.groupby("region")["amount"].agg(total="sum", orders="count", avg="mean")
    out["avg"] = out["avg"].round(2)
    return out.sort_values("total", ascending=False)

In [17]:
sales = pd.DataFrame({"region": ["N", "S", "N", "E", "S", "N"], "amount": [100, 50, 250, 90, 25, 50]})
s = summary(sales)
check("index order", lambda: list(s.index) == ["N", "E", "S"])
check("columns", lambda: list(s.columns) == ["total", "orders", "avg"])
check("values N", lambda: s.loc["N"].tolist() == [400, 3, 133.33])
check("values S", lambda: s.loc["S"].tolist() == [75, 2, 37.5])

✅ PASS - index order
✅ PASS - columns
✅ PASS - values N
✅ PASS - values S


### 🔴 Problem 9 — Merge & filter
`customers` has `id, name`; `orders` has `order_id, customer_id, amount`. Write `top_customers(customers, orders, n)` that returns a list of `(name, total_spent)` for the `n` highest spenders (highest first). Customers without orders must **not** appear.

In [18]:
def top_customers(customers, orders, n):
    merged = orders.merge(customers, left_on="customer_id", right_on="id")
    totals = merged.groupby("name")["amount"].sum().sort_values(ascending=False).head(n)
    return [(name, int(total)) for name, total in totals.items()]

In [19]:
customers = pd.DataFrame({"id": [1, 2, 3, 4], "name": ["Asha", "Ravi", "Meena", "Tom"]})
orders = pd.DataFrame({"order_id": range(1, 7), "customer_id": [1, 2, 1, 3, 2, 1], "amount": [100, 50, 200, 80, 70, 30]})
check("top 2", lambda: top_customers(customers, orders, 2) == [("Asha", 330), ("Ravi", 120)])
check("top 10 excludes customers without orders", lambda: [n for n, _ in top_customers(customers, orders, 10)] == ["Asha", "Ravi", "Meena"])

✅ PASS - top 2
✅ PASS - top 10 excludes customers without orders


### 🔴 Problem 10 — Dates and time series
`monthly_totals(df)` for a DataFrame with `date` (strings `YYYY-MM-DD`) and `value` returns a Series indexed by month string `"YYYY-MM"` with the sum of `value`, ordered chronologically. Then `busiest_weekday(df)` returns the weekday name (e.g. `"Monday"`) with the highest total value.

In [20]:
def monthly_totals(df):
    d = pd.to_datetime(df["date"])
    return df.groupby(d.dt.strftime("%Y-%m"))["value"].sum().sort_index()

def busiest_weekday(df):
    d = pd.to_datetime(df["date"])
    return df.groupby(d.dt.day_name())["value"].sum().idxmax()

In [21]:
ts = pd.DataFrame({"date": ["2025-01-06", "2025-01-13", "2025-02-03", "2025-02-04", "2025-03-10"],
                   "value": [10, 20, 5, 100, 1]})
check("monthly totals", lambda: monthly_totals(ts).to_dict() == {"2025-01": 30, "2025-02": 105, "2025-03": 1})
check("chronological", lambda: list(monthly_totals(ts).index) == ["2025-01", "2025-02", "2025-03"])
check("busiest weekday", lambda: busiest_weekday(ts) == "Tuesday")

✅ PASS - monthly totals
✅ PASS - chronological
✅ PASS - busiest weekday


### 🔴 Problem 11 — Pivot & percentages
`share_table(df)` for columns `dept, gender, salary` returns a pivot table of **average salary** with `dept` as rows and `gender` as columns (fill missing with 0, rounded to 1 decimal). `percent_of_total(s)` takes a Series and returns each value as a percentage of the sum, rounded to 1 decimal.

In [22]:
def share_table(df):
    return df.pivot_table(values="salary", index="dept", columns="gender", aggfunc="mean", fill_value=0).round(1)

def percent_of_total(s):
    return (s / s.sum() * 100).round(1)

In [23]:
emp = pd.DataFrame({"dept": ["IT", "IT", "HR", "IT"], "gender": ["F", "M", "F", "F"], "salary": [70, 60, 50, 80]})
t = share_table(emp)
check("pivot values", lambda: t.loc["IT", "F"] == 75.0 and t.loc["IT", "M"] == 60.0)
check("missing filled with 0", lambda: t.loc["HR", "M"] == 0)
check("percent_of_total", lambda: percent_of_total(pd.Series({"a": 1, "b": 3})).to_dict() == {"a": 25.0, "b": 75.0})

✅ PASS - pivot values
✅ PASS - missing filled with 0
✅ PASS - percent_of_total


### 🔴 Problem 12 — Performance: vectorise it
`slow_distance(points)` computes the pairwise distance between every pair of 2-D points using Python loops (given). Write `fast_distance(points)` that returns the **same n×n matrix** using NumPy broadcasting only.

<details><summary>💡 Hint</summary>`diff = pts[:, None, :] - pts[None, :, :]` has shape (n, n, 2); then `np.sqrt((diff ** 2).sum(axis=-1))`.</details>

In [24]:
import math

def slow_distance(points):
    n = len(points)
    return [[math.dist(points[i], points[j]) for j in range(n)] for i in range(n)]

def fast_distance(points):
    pts = np.asarray(points, dtype=float)
    diff = pts[:, None, :] - pts[None, :, :]
    return np.sqrt((diff ** 2).sum(axis=-1))

In [25]:
pts = np.random.default_rng(0).random((50, 2))
check("same result", lambda: np.allclose(fast_distance(pts), np.array(slow_distance(pts.tolist()))))
check("shape and zero diagonal", lambda: fast_distance(pts).shape == (50, 50) and np.allclose(np.diag(fast_distance(pts)), 0))
check("symmetric", lambda: np.allclose(fast_distance(pts), fast_distance(pts).T))
import time
big = np.random.default_rng(1).random((500, 2))
t0 = time.perf_counter(); fast_distance(big); fast_t = time.perf_counter() - t0
t0 = time.perf_counter(); slow_distance(big.tolist()); slow_t = time.perf_counter() - t0
check(f"faster than the Python loops (fast {fast_t:.4f}s vs slow {slow_t:.3f}s)", lambda: fast_t < slow_t)

✅ PASS - same result
✅ PASS - shape and zero diagonal
✅ PASS - symmetric
✅ PASS - faster than the Python loops (fast 0.0213s vs slow 0.082s)


---
## 🏁 Next
≥ 90 %? Go to `09_Boss_Level_challenges.ipynb`. **Bonus:** load a real CSV (e.g., Titanic or your marks) and answer 5 questions about it with pandas.